# How Did AI Feedback Affect Student Learning?

Follow-up analysis on the mid-term and end-of-term surveys from INFO 330 (n=95 and n=116).

The overview notebook covers the full response distribution and clustering. This notebook
asks targeted follow-up questions about **learning impact specifically**:

| # | Question |
|---|----------|
| 1 | Which students felt the cycle improved their understanding — and what drove that? |
| 2 | What do students say, in their own words, about how the cycle changed their learning? |
| 3 | Did the experience shift how students think about using AI for their *own* learning? |
| 4 | Did watching the full process live in lecture change their perception of AI feedback? |
| 5 | When students imagine AI feedback in other courses, what assurances do they ask for? |

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from textwrap import fill
from scipy import stats
from matplotlib.ticker import PercentFormatter, FuncFormatter
from collections import Counter

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

# ── Load both surveys ────────────────────────────────────────────────────────
mid  = pd.read_csv("./../mid-term-survey.csv")
eot  = pd.read_csv("./../end-of-term-survey.csv")

# ── Shorten end-of-term column names for convenience ────────────────────────
eot.columns = [
    "timestamp",
    "courses",
    "assignment_examples",
    "assurances",
    "shifted_own_ai_use",
    "perception_after_lecture",
    "open_to_interview",
]

# ── Likert helpers ───────────────────────────────────────────────────────────
LIKERT_ORDER = [
    "1 - Strongly disagree", "2 - Disagree", "3 - Neutral",
    "4 - Agree", "5 - Strongly agree",
]
SHORT_ORDER  = [l.split(" - ", 1)[1] for l in LIKERT_ORDER]
COLOR_MAP    = dict(zip(LIKERT_ORDER,
                   ["#B2182B", "#EF8A62", "#D9D9D9", "#67A9CF", "#2166AC"]))

def to_num(s):
    return pd.to_numeric(s.astype(str).str.extract(r"^(\d)")[0], errors="coerce")

LEARN_Q1 = "The early feedback helped me identify specific mistakes."
LEARN_Q2 = ("The resubmission for Milestone 1 and/or Milestone 2 opportunity "
            "improved my final understanding.")

mid["learning_score"] = (to_num(mid[LEARN_Q1]) + to_num(mid[LEARN_Q2])) / 2

print(f"Mid-term survey  : {len(mid)} responses")
print(f"End-of-term survey: {len(eot)} responses")
print(f"Mean composite learning score (mid-term): {mid['learning_score'].mean():.2f} / 5")

---
## Q1  Which students perceived the greatest learning gain — and what drove it?

The two Likert items most directly tied to learning are:
- *"The early feedback helped me identify specific mistakes."* (83 % agree/strongly agree)
- *"The resubmission opportunity improved my final understanding."* (96 % agree/strongly agree)

We first look at who sits in the lower-agreement tail, then ask which other perceptions
move together with reported learning gain.

In [ ]:
# ── 1a  Joint distribution: mistake-ID × understanding-improved ──────────────
q1_num = to_num(mid[LEARN_Q1])
q2_num = to_num(mid[LEARN_Q2])

ct = pd.crosstab(
    q2_num.map({1:"1",2:"2",3:"3",4:"4",5:"5"}),
    q1_num.map({1:"1",2:"2",3:"3",4:"4",5:"5"}),
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Heat-map of joint distribution
sns.heatmap(
    ct, annot=True, fmt="d", cmap="YlGnBu",
    linewidths=0.6, linecolor="white",
    ax=axes[0], cbar_kws={"label": "n students"},
)
axes[0].set_title("Joint distribution of the two learning items",
                  fontsize=13, fontweight="bold", pad=8)
axes[0].set_xlabel("Early feedback → identified mistakes (1–5)", fontsize=11)
axes[0].set_ylabel("Resubmission → improved understanding (1–5)", fontsize=11)

# Scatter with marginal rug
rng = np.random.default_rng(42)
jx = q1_num + rng.uniform(-0.18, 0.18, len(q1_num))
jy = q2_num + rng.uniform(-0.18, 0.18, len(q2_num))
axes[1].scatter(jx, jy, alpha=0.55, s=55, color="#31688E", edgecolors="white", linewidths=0.4)
rho, p = stats.spearmanr(q1_num.dropna(), q2_num[q1_num.notna()])
axes[1].set_title(f"Jittered scatter  (Spearman ρ = {rho:.2f}, p = {p:.3f})",
                  fontsize=13, fontweight="bold", pad=8)
axes[1].set_xlabel("Early feedback → identified mistakes (1–5)", fontsize=11)
axes[1].set_ylabel("Resubmission → improved understanding (1–5)", fontsize=11)
axes[1].set_xticks([1,2,3,4,5])
axes[1].set_yticks([1,2,3,4,5])
for sp in axes[1].spines.values(): sp.set_visible(False)

plt.suptitle("Learning Items: Mistake Identification vs Understanding Improvement (n=95)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Students who rated BOTH items ≥ 4: "
      f"{((q1_num>=4)&(q2_num>=4)).sum()} / {len(mid)} "
      f"({((q1_num>=4)&(q2_num>=4)).mean():.0%})")
print(f"Students who rated EITHER item ≤ 2: "
      f"{((q1_num<=2)|(q2_num<=2)).sum()} / {len(mid)} "
      f"({((q1_num<=2)|(q2_num<=2)).mean():.0%})")

In [ ]:
# ── 1b  What predicts perceived understanding improvement? ───────────────────
# Use Spearman rho on ordinal Q2 (1–5).
# Point-biserial r on the binarised outcome (q2 >= 4) is inappropriate here:
# 96% of students fall in the "positive" group, which suppresses all
# correlations and distorts predictor ranking. Spearman rho preserves the
# full ordinal variance in Q2.

ALL_LIKERT = [
    LEARN_Q1,
    "This process helps me meet the deliverable expectations.",
    "I am comfortable with AI-assisted early feedback if it is human-reviewed.",
    "I trust the challenge/appeal path when feedback seems wrong.",
    "I am confident in TA final regrading fairness and consistency.",
]

rows = []
for col in ALL_LIKERT:
    x = to_num(mid[col])
    mask = x.notna() & q2_num.notna()
    rho, p = stats.spearmanr(x[mask], q2_num[mask])
    rows.append({"predictor": col, "rho": rho, "p": p})

pred_df = pd.DataFrame(rows).sort_values("rho", ascending=True)

fig, ax = plt.subplots(figsize=(12, 4.5))
colors = ["#2166AC" if v > 0 else "#B2182B" for v in pred_df["rho"]]
bars = ax.barh([fill(p, 48) for p in pred_df["predictor"]],
               pred_df["rho"], color=colors, alpha=0.85,
               edgecolor="white", height=0.52)
for bar, (_, row) in zip(bars, pred_df.iterrows()):
    sig = "***" if row["p"]<0.001 else "**" if row["p"]<0.01 else "*" if row["p"]<0.05 else ""
    xp = row["rho"] + (0.015 if row["rho"] >= 0 else -0.015)
    ax.text(xp, bar.get_y() + bar.get_height()/2,
            f"ρ = {row['rho']:.2f}{sig}",
            va="center", ha="left" if row["rho"] >= 0 else "right", fontsize=11)
ax.axvline(0, color="#333", linewidth=0.9)
ax.set_xlabel("Spearman ρ with 'resubmission improved understanding' (ordinal 1–5)", fontsize=11)
ax.set_title("Which Perceptions Correlate with Perceived Learning Gain?",
             fontsize=13, fontweight="bold", pad=8)
ax.text(0.99, 0.04, "* p<.05  ** p<.01  *** p<.001",
        transform=ax.transAxes, ha="right", fontsize=10, color="#555")
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout()
plt.show()


---
## Q2  What do students say about how the cycle changed their learning?

Open-ended question (mid-term): *"How did the submit + quick detailed feedback +
resubmission cycle affect your learning process?"*

We use keyword-theme coding to quantify the most common ideas, then show
representative quotes per theme.

In [ ]:
LEARN_OPEN = ("How did the submit + quick detailed feedback + "
              "resubmission cycle affect your learning process?")

texts_mid = (
    mid[LEARN_OPEN].dropna().astype(str).str.strip()
)
texts_mid = texts_mid[texts_mid.str.len() >= 15].tolist()

THEMES = {
    "Identifies / fixes mistakes": re.compile(
        r"mistake|error|wrong|identify|pinpoint|spot|catch|fix", re.I),
    "Deeper understanding": re.compile(
        r"understand|concept|clarity|clearer|comprehend|insight|deeper|grasp", re.I),
    "Quick / immediate feedback": re.compile(
        r"quick|fast|immediate|timely|prompt|rapid|soon|early", re.I),
    "Iterative revision opportunity": re.compile(
        r"resubmi|revise|revision|iterate|second chance|retry|redo|rework", re.I),
    "Expectation / rubric clarity": re.compile(
        r"expect|rubric|criteria|requirement|standard|deliverable|guideline", re.I),
    "Motivation / positive affect": re.compile(
        r"motivat|confiden|encour|positive|helpful|benefit|glad|appreciat", re.I),
    "Stress or workload concern": re.compile(
        r"stress|pressure|overwhelm|workload|burden|anxious|rush|time", re.I),
}

theme_counts, theme_examples = {}, {}
for theme, pat in THEMES.items():
    matched = [t for t in texts_mid if pat.search(t)]
    theme_counts[theme]   = len(matched)
    theme_examples[theme] = matched[:3]

th_df = (pd.DataFrame({"theme": list(theme_counts),
                        "n": list(theme_counts.values())})
         .assign(pct=lambda d: d["n"] / len(texts_mid))
         .sort_values("pct", ascending=True))

palette = sns.color_palette("viridis", len(th_df))
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(th_df["theme"], th_df["pct"],
               color=palette, alpha=0.88, edgecolor="white", height=0.55)
for bar, (_, row) in zip(bars, th_df.iterrows()):
    ax.text(bar.get_width() + 0.01,
            bar.get_y() + bar.get_height()/2,
            f"{row['pct']:.0%}  (n={row['n']})",
            va="center", fontsize=11)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, 1.05)
ax.set_xlabel("Share of responses mentioning theme", fontsize=11)
ax.set_title(
    'Themes in: "How did the cycle affect your learning process?" (mid-term, n=95)',
    fontsize=13, fontweight="bold", pad=8)
ax.tick_params(axis="y", labelsize=11)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# ── Representative quotes per theme ─────────────────────────────────────────
for theme, examples in theme_examples.items():
    print(f"\n{'─'*78}")
    print(f"  {theme}")
    print(f"{'─'*78}")
    for i, ex in enumerate(examples, 1):
        print(f"  [{i}] {ex[:320]}{'…' if len(ex)>320 else ''}")

---
## Q3  Did the experience shift students' own AI use for learning?

End-of-term question (n=116): *"Did the fact that your instructor in INFO 330 used AI
for early formative feedback shift your view of your own AI use for learning in any way?"*

We code responses as positive shift, no change, or nuanced/conditional, then show
the most common reasoning.

In [ ]:
shift_texts = (
    eot["shifted_own_ai_use"].dropna().astype(str).str.strip()
)
shift_texts = shift_texts[shift_texts.str.len() >= 10].tolist()

# Lightweight sentiment coding
pos_pat  = re.compile(
    r"yes|shifted|changed.*more|more open|inspired|encouraged|positive|"
    r"made me think|opened|definitely|absolutely|useful way|differently", re.I)
neg_pat  = re.compile(
    r"\bno\b|did not|didn.t|hasn.t|not.*(shift|change|affect)|same view|"
    r"not really|not much|no change|no shift", re.I)
cond_pat = re.compile(
    r"but|however|although|though|concern|caution|careful|limit|depend|when|if", re.I)

def code(t):
    is_neg  = bool(neg_pat.search(t))
    is_pos  = bool(pos_pat.search(t))
    is_cond = bool(cond_pat.search(t))
    if is_neg and not is_pos:
        return "No shift / same view"
    if is_pos and is_cond:
        return "Positive shift, with caveats"
    if is_pos:
        return "Clear positive shift"
    if is_cond:
        return "Nuanced / conditional"
    return "Ambiguous / unclear"

codes = pd.Series([code(t) for t in shift_texts])
code_order = ["Clear positive shift", "Positive shift, with caveats",
              "Nuanced / conditional", "No shift / same view", "Ambiguous / unclear"]
code_colors = ["#2166AC", "#67A9CF", "#D9D9D9", "#EF8A62", "#B2182B"]

counts = codes.value_counts().reindex(code_order).fillna(0).astype(int)
pcts   = counts / counts.sum()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(code_order[::-1], pcts.values[::-1],
               color=code_colors[::-1], alpha=0.88,
               edgecolor="white", height=0.52)
for bar, n, p in zip(bars, counts.values[::-1], pcts.values[::-1]):
    ax.text(bar.get_width() + 0.01,
            bar.get_y() + bar.get_height()/2,
            f"{p:.0%}  (n={n})",
            va="center", fontsize=11)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, 0.80)
ax.set_xlabel("Share of end-of-term responses", fontsize=11)
ax.set_title(
    '"Did INFO 330\'s AI feedback shift your own AI use for learning?"\n'
    f'(end-of-term survey, n={len(shift_texts)})',
    fontsize=13, fontweight="bold", pad=8)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

# Common themes in the shift responses (positive group)
print("\n--- Sample 'Clear positive shift' responses ---")
pos_examples = [t for t, c in zip(shift_texts, codes) if c == "Clear positive shift"][:4]
for i, ex in enumerate(pos_examples, 1):
    print(f"  [{i}] {ex[:280]}{'…' if len(ex)>280 else ''}")

print("\n--- Sample 'No shift / same view' responses ---")
no_examples = [t for t, c in zip(shift_texts, codes) if c == "No shift / same view"][:3]
for i, ex in enumerate(no_examples, 1):
    print(f"  [{i}] {ex[:280]}{'…' if len(ex)>280 else ''}")

---
## Q4  Did watching the live lecture demo change their perception?

End-of-term question: *"After the INFO 330 instructor demonstrated the full AI-assisted
process during lecture, did that change your perception of Gen AI being used to help
provide feedback to students?"*

This question captures a distinct mechanism — transparency through demonstration, not
just disclosure in writing.

In [ ]:
2+2

In [ ]:
lec_col = eot["perception_after_lecture"]
lec_all = lec_col.fillna("").astype(str).str.strip()

# Catch "wasn't there", "wasnt there", "weren't there", "not there", etc.
not_there_pat = re.compile(
    r"wasn.?t there|was not there|not there|didn.?t attend|missed|"
    r"weren.?t there|wasnt|i were not|i wasn",
    re.I,
)

lec_not_there = lec_all[(lec_all != "") & lec_all.str.contains(not_there_pat)]
lec_valid_raw = lec_all[(lec_all != "") & ~lec_all.str.contains(not_there_pat)]
lec_texts     = lec_valid_raw[lec_valid_raw.str.len() >= 15].tolist()
lec_short     = lec_valid_raw[lec_valid_raw.str.len() < 15]
lec_empty     = (lec_all == "").sum()

n_attended    = len(lec_texts)
n_not_there   = len(lec_not_there) + len(lec_short) + lec_empty  # all non-substantive

assert n_attended + n_not_there == len(eot), \
    f"Counts don't add up: {n_attended}+{n_not_there} != {len(eot)}"

print(f"Attended & responded : {n_attended}")
print(f"Not there / too brief: {n_not_there}  ({len(lec_not_there)} matched pattern, "
      f"{len(lec_short)} short, {lec_empty} empty)")

# Code lecture-demo responses (among attendees only)
lec_pos_pat  = re.compile(
    r"yes|changed|more (comfort|open|trust|confident|positive)|enlighten|"
    r"better understand|helpful|makes sense|reassur|eye.open|impact|persuad", re.I)
lec_neg_pat  = re.compile(
    r"\bno\b|did not|didn.t|not really|not much|same|no change|no.*impact", re.I)
lec_cond_pat = re.compile(
    r"but|however|although|concern|still|worry|limit|careful|depend|if", re.I)

def code_lec(t):
    n = bool(lec_neg_pat.search(t))
    p = bool(lec_pos_pat.search(t))
    c = bool(lec_cond_pat.search(t))
    if n and not p: return "No change"
    if p and c:     return "More positive, with caveats"
    if p:           return "Clearly more positive"
    if c:           return "Nuanced / conditional"
    return "Ambiguous"

lec_codes  = pd.Series([code_lec(t) for t in lec_texts])
lec_order  = ["Clearly more positive", "More positive, with caveats",
               "Nuanced / conditional", "No change", "Ambiguous"]
lec_colors = ["#2166AC", "#67A9CF", "#D9D9D9", "#EF8A62", "#AAAAAA"]

lec_counts = lec_codes.value_counts().reindex(lec_order).fillna(0).astype(int)
lec_pcts   = lec_counts / lec_counts.sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# Left: lecture-demo perception distribution (among those who attended)
ax = axes[0]
bars = ax.barh(lec_order[::-1], lec_pcts.values[::-1],
               color=lec_colors[::-1], alpha=0.88,
               edgecolor="white", height=0.52)
for bar, n, p in zip(bars, lec_counts.values[::-1], lec_pcts.values[::-1]):
    ax.text(bar.get_width() + 0.01,
            bar.get_y() + bar.get_height()/2,
            f"{p:.0%}  (n={n})", va="center", fontsize=11)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, 0.85)
ax.set_title(f"Perception after live lecture demo\n(among those who attended, n={n_attended})",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Share of attendees", fontsize=10)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.35)

# Right: attendance breakdown — context for interpreting the left chart
ax2 = axes[1]
attend_labels = ["Attended &\nresponded", "Not there /\ntoo brief"]
attend_vals   = [n_attended, n_not_there]
attend_colors = ["#35B779", "#EF8A62"]
bars2 = ax2.bar(attend_labels, attend_vals,
                color=attend_colors, alpha=0.85, edgecolor="white", width=0.5)
for bar, n in zip(bars2, attend_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(n), ha="center", va="bottom", fontsize=13, fontweight="bold")
ax2.set_ylim(0, max(attend_vals) * 1.2)
ax2.set_ylabel("Number of students", fontsize=11)
ax2.set_title(f"Lecture demo attendance breakdown\n(end-of-term survey, n={len(eot)} total)",
              fontsize=12, fontweight="bold")
for sp in ax2.spines.values(): sp.set_visible(False)
ax2.grid(axis="y", linestyle="--", alpha=0.35)

plt.suptitle("Q4: Did Seeing the AI Process Live Change Their Perception?",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# Sample quotes
print("--- Sample 'Clearly more positive' responses ---")
pos_lec = [t for t, c in zip(lec_texts, lec_codes) if c == "Clearly more positive"][:3]
for i, ex in enumerate(pos_lec, 1):
    print(f"  [{i}] {ex[:300]}{'…' if len(ex)>300 else ''}")


---
## Q5  When students imagine AI feedback in other courses, what assurances do they ask for?

End-of-term question: *"For each class, what assurances would the instructor need to give
you so that you would feel comfortable with them using it?"*

This captures **transfer of governance awareness** — what students learned about the
conditions that make AI feedback trustworthy, expressed in a new course context.

In [ ]:
assurance_texts = (
    eot["assurances"].dropna().astype(str).str.strip()
)
assurance_texts = assurance_texts[assurance_texts.str.len() >= 10].tolist()

ASSURANCE_THEMES = {
    "Human review / oversight": re.compile(
        r"human|instructor|TA|professor|person|reviewed|overseen|double.check|"
        r"manually|someone look", re.I),
    "Transparency about AI use": re.compile(
        r"transparent|disclose|inform|told|know.*AI|aware|explain.*AI|communicate", re.I),
    "Appeal / challenge pathway": re.compile(
        r"appeal|challenge|contest|dispute|question.*feedback|disagree", re.I),
    "Accuracy / reliability assurance": re.compile(
        r"accurate|correct|reliable|valid|error.free|check.*correct|right answer", re.I),
    "Privacy / data protection": re.compile(
        r"privac|data|personal|identif|anonymi|confidential|secure|protect", re.I),
    "Feedback quality / specificity": re.compile(
        r"specific|detail|useful|quality|actionable|clear feedback|helpful feedback", re.I),
    "Resubmission / second chance": re.compile(
        r"resubmi|revise|second chance|redo|retry|opportunity", re.I),
}

assur_counts = {}
assur_examples = {}
for theme, pat in ASSURANCE_THEMES.items():
    matched = [t for t in assurance_texts if pat.search(t)]
    assur_counts[theme]   = len(matched)
    assur_examples[theme] = matched[:2]

assur_df = (pd.DataFrame({"theme": list(assur_counts),
                           "n": list(assur_counts.values())})
            .assign(pct=lambda d: d["n"] / len(assurance_texts))
            .sort_values("pct", ascending=True))

palette_a = sns.color_palette("mako", len(assur_df))
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(assur_df["theme"], assur_df["pct"],
               color=palette_a, alpha=0.88, edgecolor="white", height=0.52)
for bar, (_, row) in zip(bars, assur_df.iterrows()):
    ax.text(bar.get_width() + 0.01,
            bar.get_y() + bar.get_height()/2,
            f"{row['pct']:.0%}  (n={row['n']})",
            va="center", fontsize=11)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, 1.05)
ax.set_xlabel("Share of end-of-term responses mentioning assurance type", fontsize=11)
ax.set_title(
    '"What assurances would you need to feel comfortable?"\n'
    f'(Transfer to other courses — end-of-term, n={len(assurance_texts)})',
    fontsize=13, fontweight="bold", pad=8)
ax.tick_params(axis="y", labelsize=11)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

print("\n--- Sample quotes per assurance theme ---")
for theme, examples in assur_examples.items():
    print(f"\n  {theme}")
    for i, ex in enumerate(examples, 1):
        print(f"    [{i}] {ex[:240]}{'…' if len(ex)>240 else ''}")

---
## Summary: Key Learning-Impact Findings

In [ ]:
q1_agree = (to_num(mid[LEARN_Q1]) >= 4).mean()
q2_agree = (to_num(mid[LEARN_Q2]) >= 4).mean()
both_hi  = ((to_num(mid[LEARN_Q1])>=4) & (to_num(mid[LEARN_Q2])>=4)).mean()

shift_codes   = pd.Series([code(t) for t in shift_texts])
shift_pos_pct = shift_codes.isin(
    ["Clear positive shift", "Positive shift, with caveats"]).mean()
lec_pos_pct   = lec_codes.isin(
    ["Clearly more positive", "More positive, with caveats"]).mean()

print("=" * 70)
print("  KEY LEARNING-IMPACT FINDINGS")
print("=" * 70)
print(f"""
MID-TERM SURVEY (n=95, right after Milestone 2 AI feedback)

  {q1_agree:.0%} agreed the early feedback helped identify specific mistakes.
  {q2_agree:.0%} agreed the resubmission opportunity improved their understanding.
  {both_hi:.0%} rated both items ≥ 4 (Agree).

  Strongest correlates of perceived understanding improvement (Spearman ρ):
    1. 'Process helped meet deliverable expectations'  ρ = 0.67 ***
    2. 'Early feedback helped identify mistakes'       ρ = 0.46 ***
    3. 'Comfortable with AI-assisted feedback'         ρ = 0.28 **
    Trust in appeal path and TA regrading confidence are not significantly
    correlated — governance items matter for acceptance, not for perceived
    learning gain directly.

  Top open-ended themes in learning descriptions:
    Mistake identification, iterative revision, and quick feedback
    dominate. Stress/workload appears but is a minority concern.

END-OF-TERM SURVEY (n=116, after completing the full course)

  {shift_pos_pct:.0%} reported a positive shift in how they think about
    using AI for their own learning (with or without caveats).

  {n_attended} of {len(eot)} students attended the live lecture demo and responded.
  Of those, {lec_pos_pct:.0%} said it made them more positive about AI feedback.
  (Note: lecture-demo % is among attendees only — not comparable to the
   {shift_pos_pct:.0%} figure above, which covers all end-of-term respondents.)

  Top assurances students asked for in other courses:
    Human review, transparency about AI use, and accuracy/reliability —
    the same governance features built into the INFO 330 model.
""")
